# 8. Exercise-Dependent Counting Colab


## Reason, Approach, Result Interpretation

**Why this stage exists**

The generic pose, RGB, and simple multimodal branches did not produce one strong shared counter across the supported exercises. The current evidence is exercise-dependent instead:

- `squat`: dedicated pose branch remains clearly best
- `push_up`: RGB branch is clearly better than pose
- `pull_up`: mixed, but pose is the safer current choice when near-exact counting is prioritized

**Approach**

This stage does not train a new model. It builds a routed counting surface from the strongest current branch per supported exercise and stitches the row-level predictions into one combined artifact.

The first routed baseline uses:

- `squat` -> `squat_tcn_l1_channels96`
- `pull_up` -> `pose_count_tcn_pull_up_seq192`
- `push_up` -> `rgb_count_tcn_push_up_seq128`

This route is intentionally pragmatic rather than generic. It is meant to provide a usable counting surface for selected videos while keeping the representation choice honest.

**How to interpret the result**

If the routed system gives clearly better supported-exercise counting than the generic shared branches, then the project has moved from representation diagnosis to a usable exercise-dependent prototype. If it is still weak, then the remaining limitation is not only representation choice, but data/task difficulty itself.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

ROUTER_REL = Path('artifacts/3_Modeling/build_routed_count_predictions.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')

def sync_drive_file(rel: Path) -> None:
    src = CODE_ROOT / rel
    dst = DRIVE_PROJECT_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src.exists() and not dst.exists():
        print(f'[sync] missing both copies: {rel}')
        return
    if not src.exists():
        print(f'[sync] keeping Drive copy (no /content source): {rel}')
        return
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f'[sync] copied /content -> Drive (Drive missing): {rel}')
        return
    if src.stat().st_mtime > dst.stat().st_mtime + 1.0:
        shutil.copy2(src, dst)
        print(f'[sync] copied newer /content -> Drive: {rel}')
    else:
        print(f'[sync] keeping Drive copy (newer or equal): {rel}')

for rel in [ROUTER_REL, COMPARE_REL]:
    sync_drive_file(rel)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
POSE_SEQUENCE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'
ROUTED_OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / 'routed_exercise_dependent_counting'

print('POSE_SEQUENCE_INDEX =', POSE_SEQUENCE_INDEX, POSE_SEQUENCE_INDEX.exists())
print('ROUTER_PRESENT =', (DRIVE_PROJECT_ROOT / ROUTER_REL).exists())


## Routing Preset


In [ ]:
import pandas as pd

ROUTES = [
    {
        'exercise': 'squat',
        'run_name': 'squat_tcn_l1_channels96',
        'reason': 'Best dedicated squat pose baseline by a large margin.',
    },
    {
        'exercise': 'pull_up',
        'run_name': 'pose_count_tcn_pull_up_seq192',
        'reason': 'Slightly higher MAE than stronger RGB, but much better Within-1.',
    },
    {
        'exercise': 'push_up',
        'run_name': 'rgb_count_tcn_push_up_seq128',
        'reason': 'Best current push_up branch in both practical and research terms.',
    },
]

route_df = pd.DataFrame(ROUTES)
display(route_df)


## Build Routed Predictions


In [ ]:
import subprocess

cmd = [
    'python', '-u', str(DRIVE_PROJECT_ROOT / ROUTER_REL),
    '--project-dir', str(DRIVE_PROJECT_ROOT),
    '--index-csv', str(POSE_SEQUENCE_INDEX),
    '--output-dir', str(ROUTED_OUTPUT_DIR),
]
for row in ROUTES:
    cmd.extend(['--route', f"{row['exercise']}={row['run_name']}"])

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## Routed Metric Review


In [ ]:
import json
import pandas as pd

summary_path = ROUTED_OUTPUT_DIR / 'routed_metrics_summary.json'
routing_csv = ROUTED_OUTPUT_DIR / 'routing_summary.csv'
pred_csv = ROUTED_OUTPUT_DIR / 'routed_predictions.csv'

with open(summary_path, 'r', encoding='utf-8') as f:
    routed_summary = json.load(f)

routing_df = pd.read_csv(routing_csv)
routed_df = pd.read_csv(pred_csv)

print('Per-exercise valid metrics:')
per_ex_valid = []
for exercise, split_metrics in sorted(routed_summary['per_exercise_split_metrics'].items()):
    valid_metrics = split_metrics.get('valid')
    if valid_metrics:
        per_ex_valid.append({
            'exercise': exercise,
            'valid_mae': valid_metrics['mae'],
            'valid_rmse': valid_metrics['rmse'],
            'valid_within_1': valid_metrics['within_1'],
        })
display(pd.DataFrame(per_ex_valid).sort_values('exercise'))

print('\nSplit metrics:')
print(json.dumps(routed_summary['split_metrics'], indent=2))

print('\nRouting summary:')
display(routing_df.sort_values('exercise'))


## Routed Baseline Comparison


In [ ]:
import subprocess
import json
import pandas as pd

pred_csv = ROUTED_OUTPUT_DIR / 'routed_predictions.csv'
summary_json = ROUTED_OUTPUT_DIR / 'baseline_comparison_summary.json'
rows_csv = ROUTED_OUTPUT_DIR / 'baseline_comparison_rows.csv'

cmd = [
    'python', str(DRIVE_PROJECT_ROOT / COMPARE_REL),
    '--index-csv', str(POSE_SEQUENCE_INDEX),
    '--predictions-csv', str(pred_csv),
    '--output-json', str(summary_json),
    '--output-csv', str(rows_csv),
]
print('Comparing:', ' '.join(cmd))
subprocess.run(cmd, check=True)

with open(summary_json, 'r', encoding='utf-8') as f:
    baseline_summary = json.load(f)

print(json.dumps({
    'model_metrics': baseline_summary['model_metrics'],
    'baseline_metrics': baseline_summary['baseline_metrics'],
    'delta_vs_baseline': baseline_summary['delta_vs_baseline'],
    'row_level': baseline_summary['row_level'],
}, indent=2))


## Video Lookup


In [ ]:
import pandas as pd

LOOKUP_NAMES = [
    # 'train3898.mp4',
]

routed_df = pd.read_csv(ROUTED_OUTPUT_DIR / 'routed_predictions.csv')
lookup_df = routed_df[routed_df['name'].isin(LOOKUP_NAMES)].copy()
if lookup_df.empty:
    print('No lookup rows selected yet. Add video names to LOOKUP_NAMES and rerun this cell.')
else:
    display(lookup_df.sort_values(['type', 'split', 'name']))
